# Phase 2 — Per-cell degradation parameter identification

This notebook drives `phase2_de_fit.py` across the 13-cell training cohort.

**What Phase 2 does:** for each cohort cell, run a differential-evolution fit of 5 PyBaMM degradation parameters against the measured Longterm SoH trajectory. The fit uses:

1. **Per-cell BOL overrides** from `configs/bol_params/{make}_{cell}.yaml` (Phase 1 outputs — stoichiometry, R0, D_s)
2. **Per-cell experiment protocol** from `configs/cohort_experiment_protocols.yaml` (real Athena-derived CC/CV window per cell — this is the key correctness change vs. the earlier EVE-0008-only work)
3. **RMSE loss in percentage points** over the FULL measured horizon (no 150-cycle cap)

**Fitted parameters (5):**

| Symbol | PyBaMM key | Encoding | Bounds |
|---|---|---|---|
| `k_SEI`          | SEI kinetic rate constant [m.s-1]                          | log10 | [-13, -10] |
| `V_SEI`          | SEI partial molar volume [m3.mol-1]                        | linear | [8e-5, 2e-4] |
| `D_SEI_solvent`  | SEI solvent diffusivity [m2.s-1]                           | log10 | [-22, -18] |
| `k_plating`      | Lithium plating kinetic rate constant [m.s-1]              | log10 | [-12, -9] |
| `k_LAM_negative` | Negative electrode LAM constant proportional term [s-1]    | log10 | [-10, -7] |

**DE budget:** popsize=6/param → 30 individuals/gen · ~7 gens · ~210 evaluations · tol=1e-3.

**Outputs:** `configs/deg_params/{make}_{cell}.yaml` per cell + `configs/deg_params/_aggregate.md`.

## 1. Setup

In [ ]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')

HERE = Path('/home/hj/Desktop/PINNs/Voltaris/Data_Exploration')
PROJECT = Path('/home/hj/Desktop/PINNs')
CONFIGS = PROJECT / 'configs'
DEG_OUT_DIR = CONFIGS / 'deg_params'
DEG_OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(HERE))
import phase2_de_fit as p2

# Cohort manifest
manifest = yaml.safe_load((HERE / 'training_cohort.yaml').read_text())
COHORT = {k: [str(c).zfill(4) for c in v] for k, v in manifest['cohort'].items()}
CELL_LIST = [(m, c) for m, cells in COHORT.items() for c in cells]
print(f'Cohort: {len(CELL_LIST)} cells')
for make, cells in COHORT.items():
    print(f'  {make}: {cells}')

MAKE_COLOR = {'CALB': '#c94a3c', 'EVE': '#4dab5c', 'REPT': '#3c7cc9'}

## 2. Per-cell fit loop

**Smoke-test mode is ON by default**. Set `SMOKE_TEST_MODE = False` only when you're ready to kick the full 13-cell run — a single cell takes ~30–60 minutes wall-clock with `workers=-1`, so the full cohort is a multi-hour job.

In [ ]:
SMOKE_TEST_MODE = True             # <-- flip to False for the full cohort
SMOKE_CELL = ('EVE', '0008')       # single cell for the smoke test
SMOKE_N_CYCLES_OVERRIDE = 30       # truncate horizon (only in smoke mode)
SMOKE_N_EVALS = 60                 # tiny DE budget for the smoke test

FULL_N_EVALS = 210                 # brief-signed: popsize×5 × ~7 gens
FULL_WORKERS = -1                  # brief: use all cores (host may need cap)

if SMOKE_TEST_MODE:
    run_list = [SMOKE_CELL]
    n_evals = SMOKE_N_EVALS
    n_cycles_override = SMOKE_N_CYCLES_OVERRIDE
    workers = 1
    print(f'SMOKE MODE — 1 cell ({SMOKE_CELL}), '
          f'{n_evals} evals, {n_cycles_override} cycles.')
else:
    run_list = CELL_LIST
    n_evals = FULL_N_EVALS
    n_cycles_override = None
    workers = FULL_WORKERS
    print(f'FULL COHORT MODE — {len(run_list)} cells, '
          f'{n_evals} evals/cell, workers={workers}. This will take hours.')

In [ ]:
results = {}
for i, (make, cell) in enumerate(run_list, 1):
    key = f'{make}_{cell}'
    print(f'\n[{i}/{len(run_list)}] {key} ...', flush=True)
    t0 = time.time()
    try:
        res = p2.identify_cell_deg(
            make, cell,
            n_evaluations=n_evals,
            workers=workers,
            n_cycles_override=n_cycles_override,
            verbose=True,
        )
        out_path = DEG_OUT_DIR / f'{key}.yaml'
        p2.save_yaml(res, out_path)
        results[key] = res
        print(f'  wrote: {out_path.relative_to(PROJECT)}  '
              f'RMSE={res["best_rmse_pp"]:.3f} pp  '
              f'wall={time.time()-t0:.1f}s', flush=True)
    except Exception as ex:
        print(f'  FAILED: {type(ex).__name__}: {ex}', flush=True)
        results[key] = None
print(f'\nDone. {sum(1 for v in results.values() if v is not None)}/{len(results)} succeeded.')

## 3. Aggregate table

One row per cell: RMSE, fitted theta, identifiability flags.

In [ ]:
def _row_from_result(key, res):
    row = dict(cell_key=key, make=res['make'], cell=res['cell'],
               protocol_id=res['protocol_id'],
               n_cycles=res['n_cycles_fitted'],
               RMSE_pp=res['best_rmse_pp'],
               wall_time_s=res['de']['wall_time_s'],
               de_evals=res['de']['n_evaluations'],
               de_success=res['de']['success'])
    for name, meta in res['fitted_params'].items():
        row[name] = meta['value']
        ident = meta.get('identifiability') or {}
        row[f'{name}_well_id'] = ident.get('well_identified')
        row[f'{name}_span'] = ident.get('span_of_full_range')
    return row

rows = [_row_from_result(k, r) for k, r in results.items() if r is not None]
if rows:
    agg = pd.DataFrame(rows)
    round_map = {'RMSE_pp': 3, 'wall_time_s': 1}
    for s in p2.THETA_SPEC:
        round_map[s['name']] = 4
        round_map[f"{s['name']}_span"] = 3
    display(agg.round(round_map))
else:
    print('No successful results yet.')

## 4. Per-cell SoH overlay

Measured (rolling-median-smoothed) vs. simulated-with-fitted-θ per cell.

In [ ]:
def _sim_with_result(res):
    ctx = p2.load_cell_context(res['make'], res['cell'])
    pv = p2.build_pybamm_parameters(ctx['bol'])
    theta = {name: res['fitted_params'][name]['value']
             for name in [s['name'] for s in p2.THETA_SPEC]}
    pv = p2.apply_deg_params(pv, **theta)
    n_meas = int(res['n_cycles_fitted'])
    soh_sim = p2.simulate_soh_trajectory(
        pv, ctx['experiment_steps'],
        n_cycles=n_meas,
        nominal_capacity_Ah=ctx['nominal_capacity_Ah'],
    )
    meas = np.asarray(ctx['measured_soh_array'], dtype=float)[:n_meas]
    meas_norm = meas / meas[0] if meas.size and meas[0] > 0 else meas
    return soh_sim, meas_norm, ctx

def _smoothed(y, w=7):
    return pd.Series(y).rolling(w, center=True, min_periods=3).median().values

def plot_overlays(res_dict):
    ok = [(k, r) for k, r in res_dict.items() if r is not None]
    if not ok:
        print('No results to plot.')
        return
    ncols = min(2, len(ok))
    nrows = int(np.ceil(len(ok) / ncols))
    fig = make_subplots(rows=nrows, cols=ncols,
                         subplot_titles=[k for k, _ in ok],
                         vertical_spacing=0.10, horizontal_spacing=0.09)
    for i, (k, r) in enumerate(ok):
        row = i // ncols + 1; col = i % ncols + 1
        soh_sim, meas_norm, _ = _sim_with_result(r)
        color = MAKE_COLOR.get(r['make'], '#333')
        x = np.arange(1, len(meas_norm) + 1)
        fig.add_trace(go.Scatter(x=x, y=_smoothed(meas_norm), name='measured',
                                  legendgroup='measured',
                                  showlegend=(i == 0),
                                  line=dict(color='#333', width=2),
                                  mode='lines'), row=row, col=col)
        xs = np.arange(1, len(soh_sim) + 1)
        fig.add_trace(go.Scatter(x=xs, y=soh_sim, name='simulated (fitted θ)',
                                  legendgroup='sim',
                                  showlegend=(i == 0),
                                  line=dict(color=color, width=2, dash='dash'),
                                  mode='lines'), row=row, col=col)
        rmse = r['best_rmse_pp'] or float('nan')
        fig.add_annotation(text=f'RMSE={rmse:.2f} pp',
                            xref=f'x{i+1 if i>0 else ""} domain',
                            yref=f'y{i+1 if i>0 else ""} domain',
                            x=0.98, y=0.05, showarrow=False,
                            font=dict(size=10, color='#666'),
                            row=row, col=col)
        fig.update_xaxes(title_text='cycle', row=row, col=col,
                          gridcolor='rgba(0,0,0,0.08)')
        fig.update_yaxes(title_text='SoH / SoH_cy1', row=row, col=col,
                          gridcolor='rgba(0,0,0,0.08)')
    fig.update_layout(title='Measured vs simulated SoH (fitted θ) per cell',
                       plot_bgcolor='white',
                       height=340 * nrows + 60, width=1050,
                       margin=dict(l=60, r=40, t=70, b=50))
    fig.show()

plot_overlays(results)

## 5. Cross-cohort θ distribution

One panel per parameter — cells positioned along the search axis, coloured by make. Dashed lines = search bounds.

In [ ]:
def plot_theta_distributions(res_dict):
    ok = [r for r in res_dict.values() if r is not None]
    if not ok:
        print('No results.')
        return
    specs = p2.THETA_SPEC
    ncols = min(3, len(specs))
    nrows = int(np.ceil(len(specs) / ncols))
    fig = make_subplots(rows=nrows, cols=ncols,
                         subplot_titles=[s['name'] for s in specs],
                         vertical_spacing=0.15, horizontal_spacing=0.10)
    makes = ['CALB', 'EVE', 'REPT']
    y_pos = {m: i for i, m in enumerate(makes)}
    for si, spec in enumerate(specs):
        row = si // ncols + 1; col = si % ncols + 1
        for m in makes:
            xs, cells = [], []
            for r in ok:
                if r['make'] != m: continue
                v = r['fitted_params'][spec['name']]['value']
                if not np.isfinite(v): continue
                xs.append(np.log10(v) if spec['encoding'] == 'log10' else v)
                cells.append(r['cell'])
            if not xs: continue
            ys = np.full(len(xs), y_pos[m], dtype=float) + \
                 np.random.default_rng(hash(m+spec['name']) & 0xFFFFFFFF).uniform(-0.15, 0.15, len(xs))
            fig.add_trace(go.Scatter(x=xs, y=ys, mode='markers+text',
                                      marker=dict(size=10, color=MAKE_COLOR[m],
                                                   line=dict(color='white', width=1.2)),
                                      text=cells, textposition='top center',
                                      textfont_size=8, showlegend=False,
                                      hovertemplate=f'<b>{m} · %{{text}}</b><br>'
                                                     f'{spec["name"]}: %{{x}}<extra></extra>'),
                          row=row, col=col)
        # Bounds
        lo, hi = spec['bounds']
        fig.add_vline(x=lo, row=row, col=col,
                       line=dict(color='grey', dash='dash', width=0.8))
        fig.add_vline(x=hi, row=row, col=col,
                       line=dict(color='grey', dash='dash', width=0.8))
        axis_label = f"log10({spec['name']})" if spec['encoding']=='log10' else spec['name']
        fig.update_xaxes(title_text=axis_label, row=row, col=col,
                          gridcolor='rgba(0,0,0,0.08)')
        fig.update_yaxes(tickmode='array',
                          tickvals=list(y_pos.values()),
                          ticktext=list(y_pos.keys()),
                          range=[-0.6, 2.6],
                          gridcolor='rgba(0,0,0,0.05)', row=row, col=col)
    fig.update_layout(title='Fitted θ distribution across cohort',
                       plot_bgcolor='white',
                       height=320 * nrows + 60, width=1150,
                       margin=dict(l=60, r=40, t=70, b=50))
    fig.show()

plot_theta_distributions(results)

## 6. Aggregate markdown report

Writes `configs/deg_params/_aggregate.md` — one-line-per-cell summary + per-parameter identifiability flags.

In [ ]:
def _fmt(v, prec=3):
    if v is None or (isinstance(v, float) and not np.isfinite(v)): return 'nan'
    return f'{v:.{prec}g}'

def build_aggregate_md(res_dict):
    ok = [r for r in res_dict.values() if r is not None]
    lines = []
    lines.append('# Phase 2 — per-cell degradation parameter fit')
    lines.append('')
    lines.append(f'Generated: {pd.Timestamp.utcnow().isoformat()}Z  ')
    lines.append(f'Cells fitted: **{len(ok)}** / {len(res_dict)}')
    lines.append('')
    lines.append('## Fit summary')
    lines.append('')
    lines.append('| cell | protocol | n_cy | RMSE (pp) | wall (s) | DE ok |')
    lines.append('|---|---|---|---|---|---|')
    for r in ok:
        lines.append(f"| {r['make']}_{r['cell']} | {r['protocol_id']} | "
                     f"{r['n_cycles_fitted']} | {_fmt(r['best_rmse_pp'])} | "
                     f"{_fmt(r['de']['wall_time_s'], 4)} | {r['de']['success']} |")
    lines.append('')
    lines.append('## Fitted θ (physical units)')
    lines.append('')
    names = [s['name'] for s in p2.THETA_SPEC]
    lines.append('| cell | ' + ' | '.join(names) + ' |')
    lines.append('|' + '---|' * (len(names) + 1))
    for r in ok:
        vals = [_fmt(r['fitted_params'][n]['value'], 3) for n in names]
        lines.append(f"| {r['make']}_{r['cell']} | " + ' | '.join(vals) + ' |')
    lines.append('')
    lines.append('## Identifiability (top-10% span / bound width; well_id iff <0.30)')
    lines.append('')
    lines.append('| cell | ' + ' | '.join(f'{n} span' for n in names) + ' |')
    lines.append('|' + '---|' * (len(names) + 1))
    for r in ok:
        cells_txt = []
        for n in names:
            ident = r['fitted_params'][n].get('identifiability') or {}
            span = ident.get('span_of_full_range')
            flag = ident.get('well_identified')
            mark = '✓' if flag else '·'
            cells_txt.append(f'{_fmt(span, 2)} {mark}')
        lines.append(f"| {r['make']}_{r['cell']} | " + ' | '.join(cells_txt) + ' |')
    return '\n'.join(lines) + '\n'

md = build_aggregate_md(results)
out_md = DEG_OUT_DIR / '_aggregate.md'
out_md.write_text(md)
print(f'Wrote: {out_md}')
display(Markdown(md))